# Matching model, step 1: zero-shot, then fine-tuned bi-encoder

Run cells **top to bottom, one at a time** on a **T4 GPU** (Runtime > Change runtime type).

Idea: instead of a classifier with 180 outputs, embed the sentence and each technique's description, and rank all techniques by similarity (**733** = all 737 ATT&CK v12 Enterprise labels minus 4 revoked ids whose description is identical to the active id that replaced them). Unseen techniques can then be predicted too.

| Cell | What it does |
|---|---|
| 1 | Paths, imports; downloads `ttp-desc-mappings.csv` if missing |
| 2 | Data, label space, metrics (paper-style micro P@1 / R@1, MRR@3) |
| 3 | Model + training code (defines only) |
| 4 | Zero-shot: no training at all |
| 5 | Fine-tune on TRAM train (about 20-30 min) |
| 6 | Results table vs the baselines |

Outputs go to `run_match_1`.

In [ ]:
# Cell 1 - setup, paths, imports
import os, ast, json, random, re, math, warnings, urllib.request
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_DIR = "/content/drive/MyDrive/KDU/Thesis/Dataset"
OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/run_match_1"          # new folder for this experiment
os.makedirs(OUT_ROOT, exist_ok=True)

# Technique descriptions (737 ATT&CK v12 labels) come from the paper's repo. Download once if missing.
DESC_PATH = f"{DATA_DIR}/ttp-desc-mappings.csv"
if not os.path.exists(DESC_PATH):
    url = "https://raw.githubusercontent.com/tumeteor/mitre-ttp-mapping/main/datasets/ttp-desc-mappings.csv"
    urllib.request.urlretrieve(url, DESC_PATH)
    print("downloaded", DESC_PATH)

BASE_MODEL = "BAAI/bge-base-en-v1.5"      # embedding model (try "BAAI/bge-large-en-v1.5" later if memory allows)
print("OUT_ROOT =", OUT_ROOT)

In [ ]:
# Cell 2 - data, label space (737 ATT&CK v12 Enterprise labels minus 4 revoked duplicate ids) and metrics
def load_split(name):
    df = pd.read_csv(f"{DATA_DIR}/tram_{name}.tsv", sep="\t").rename(columns={"text1": "text"})
    df["y"] = df["labels"].map(lambda x: sorted({s.lower() for s in ast.literal_eval(x)}))   # lowercase ids, de-duplicated
    return df[["text", "y"]]

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)
assert len(train) == 3468, len(train)

desc = pd.read_csv(DESC_PATH, sep="\t")
desc["ttp"] = desc["ttp"].str.lower().str.strip()
desc["description"] = (desc["description"].fillna("").str.replace(r"\s+", " ", regex=True)
                       .str.replace(r"\bcitation:\s*", "", regex=True).str.strip())   # drop the literal 'citation:' markers
desc = desc.drop_duplicates("ttp").reset_index(drop=True)

# Four REVOKED v12 technique ids have a description identical to the active id that replaced them
# (checked against the official ATT&CK v12.0 data): t1060->t1547.001, t1107->t1070.004, t1151->t1036.006,
# t1188->t1090.003. They would tie with the correct label, so we remove the 4 old ids. Nothing else is removed.
drop = {"t1060", "t1107", "t1151", "t1188"}
assert drop <= set(desc["ttp"]), "drop list does not match the description file"
print("removed revoked duplicate ids:", sorted(drop))
desc = desc[~desc["ttp"].isin(drop)].reset_index(drop=True)
LABELS = desc["ttp"].tolist(); LABEL_TEXTS = desc["description"].tolist()
L = len(LABELS); lab2i = {l: i for i, l in enumerate(LABELS)}
print("labels in the matching space:", L)

def to_Y(df):
    Y = np.zeros((len(df), L), dtype=np.int64)
    missing = set()
    for i, ys in enumerate(df["y"]):
        for y in ys:
            if y in lab2i: Y[i, lab2i[y]] = 1
            else: missing.add(y)
    if missing: print("WARNING labels not in description file:", sorted(missing))
    return Y

Y_train, Y_dev, Y_test = to_Y(train), to_Y(dev), to_Y(test)
SEEN = Y_train.sum(axis=0) > 0                       # labels that occur in TRAM train
print(f"train={len(train)} dev={len(dev)} test={len(test)} | seen labels={SEEN.sum()} | "
      f"test gold labels={Y_test.sum()} (unseen-in-train: {int((Y_test.astype(bool) & ~SEEN[None, :]).sum())})")
assert (Y_train.sum(axis=1) > 0).all(), "a train row has no known label"

# parent technique of each label (for technique-level P@1)
PARENTS = [l.split(".")[0] for l in LABELS]
_up = sorted(set(PARENTS)); PAR_IDX = np.array([_up.index(p) for p in PARENTS])
PARMAT = np.zeros((L, len(_up))); PARMAT[np.arange(L), PAR_IDX] = 1

def rank_metrics(S, Y):
    """S: scores (rows x 737). Paper-style micro metrics over ALL gold labels (unseen ones included)."""
    n = len(S); top = np.argsort(-S, axis=1)[:, :5]
    hits = np.take_along_axis(Y, top, axis=1); total = Y.sum()
    h3 = hits[:, :3]
    first = np.where(h3.any(axis=1), h3.argmax(axis=1) + 1, np.inf)
    intop = np.zeros(Y.shape, dtype=bool); intop[np.arange(n)[:, None], top] = True
    gold = Y.astype(bool); un = gold & ~SEEN[None, :]; se = gold & SEEN[None, :]
    ypar = (Y @ PARMAT) > 0
    tech_hit = ypar[np.arange(n), PAR_IDX[top[:, 0]]]
    return {"P@1": float(hits[:, 0].mean()), "R@1": float(hits[:, 0].sum() / total),
            "MRR@3": float(np.mean(1.0 / first)), "R@5": float(hits.sum() / total),
            "tech_P@1": float(tech_hit.mean()),
            "seen_R@5": float((intop & se).sum() / max(se.sum(), 1)),
            "unseen_R@5": float((intop & un).sum() / max(un.sum(), 1)),
            "n_unseen_pairs": int(un.sum())}

In [ ]:
# Cell 3 - bi-encoder model + training code
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class Matcher(nn.Module):
    """One shared encoder for sentences and label descriptions; cosine similarity."""
    def __init__(self, name, pooling="cls"):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name); self.pooling = pooling
    def forward(self, **enc):
        h = self.enc(**enc).last_hidden_state
        if self.pooling == "cls":
            v = h[:, 0]
        else:
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype); v = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return F.normalize(v.float(), dim=-1)

def tok_batch(tok, texts, max_len):
    enc = tok(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask", "token_type_ids")}

@torch.no_grad()
def embed_all(model, tok, texts, max_len, bs=64, amp=True):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            out.append(model(**tok_batch(tok, texts[i:i + bs], max_len)).cpu())
    return torch.cat(out).numpy()

def score_split(model, tok, texts, label_emb, text_max_len=128):
    return embed_all(model, tok, list(texts), text_max_len) @ label_emb.T       # rows x labels cosine scores

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def run_eval(model, tok, label_max_len_eval=256):
    label_emb = embed_all(model, tok, LABEL_TEXTS, label_max_len_eval)
    S_dev = score_split(model, tok, dev["text"], label_emb); S_te = score_split(model, tok, test["text"], label_emb)
    return S_dev, S_te, rank_metrics(S_dev, Y_dev), rank_metrics(S_te, Y_test)

def train_matcher(model_name, out_dir, seed=42, epochs=8, patience=3, batch_size=16, lr=2e-5, n_neg=64,
                  scale=30.0, text_max_len=128, label_max_len_train=128, label_max_len_eval=256,
                  pooling="cls", use_amp=True, neg_pool="all"):
    set_seed(seed); os.makedirs(out_dir, exist_ok=True); rng = np.random.default_rng(seed)
    amp_on = use_amp and device.type == "cuda"
    tok = AutoTokenizer.from_pretrained(model_name)
    model = Matcher(model_name, pooling).to(device)
    N = len(train); texts = train["text"].tolist(); label_texts = np.array(LABEL_TEXTS, dtype=object)
    pos_lists = [np.where(Y_train[i])[0] for i in range(N)]
    pool = np.arange(L) if neg_pool == "all" else np.where(SEEN)[0]
    steps_per_epoch = math.ceil(N / batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps_per_epoch * epochs), steps_per_epoch * epochs)
    scaler = torch.amp.GradScaler(enabled=amp_on)
    print(f"[{model_name}] seed={seed} epochs<={epochs} bs={batch_size} lr={lr} n_neg={n_neg} scale={scale} neg_pool={neg_pool}")

    best, best_ep, best_state, bad, hist = -1.0, 0, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); perm = rng.permutation(N); tot = 0.0
        for b in range(0, N, batch_size):
            idx = perm[b:b + batch_size]
            gold_u = np.unique(np.concatenate([pos_lists[i] for i in idx]))
            cand = np.setdiff1d(pool, gold_u)
            negs = rng.choice(cand, size=min(n_neg, len(cand)), replace=False)
            U = np.concatenate([gold_u, negs])                                  # label set for this step
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_on):
                et = model(**tok_batch(tok, [texts[i] for i in idx], text_max_len))
                eu = model(**tok_batch(tok, label_texts[U].tolist(), label_max_len_train))
            S = (et @ eu.T) * scale                                             # batch x |U|, fp32
            pos = torch.tensor(Y_train[idx][:, U].astype(bool), device=device)
            neg_lse = torch.logsumexp(S.masked_fill(pos, -1e4), dim=1)          # log-sum-exp over negatives only
            rows = pos.nonzero()[:, 0]
            loss = F.softplus(neg_lse[rows] - S[pos]).mean()                    # = -log softmax of each positive vs negatives
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(idx)
        _, _, md, _ = run_eval(model, tok, label_max_len_eval)
        hist.append({"epoch": ep, "train_loss": tot / N, **{f"dev_{k}": v for k, v in md.items()}})
        print(f"  epoch {ep:2d}: train_loss={tot / N:.4f} dev P@1={md['P@1']:.4f} MRR@3={md['MRR@3']:.4f} "
              f"R@5={md['R@5']:.4f} unseenR@5={md['unseen_R@5']:.3f}")
        if md["MRR@3"] > best:                                                  # select epoch by DEV MRR@3
            best, best_ep, bad = md["MRR@3"], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop (no dev improvement for {patience} epochs)"); break

    model.load_state_dict(best_state)
    S_dev, S_te, md, mt = run_eval(model, tok, label_max_len_eval)
    np.save(f"{out_dir}/dev_scores.npy", S_dev); np.save(f"{out_dir}/test_scores.npy", S_te)
    json.dump({"model_name": model_name, "seed": seed, "best_epoch": best_ep, "dev": md, "test": mt, "history": hist,
               "config": dict(epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, n_neg=n_neg, scale=scale,
                              text_max_len=text_max_len, label_max_len_train=label_max_len_train, pooling=pooling,
                              neg_pool=neg_pool)}, open(f"{out_dir}/metrics.json", "w"), indent=2)
    print("  TEST:", {k: round(v, 4) if isinstance(v, float) else v for k, v in mt.items()})
    print(f"  saved to {out_dir} (best epoch {best_ep})")
    del model, opt, best_state; import gc; gc.collect(); torch.cuda.empty_cache()
    return mt

In [ ]:
# Cell 4 - ZERO-SHOT: pretrained embedder, no training at all (sentence vs technique description)
tok0 = AutoTokenizer.from_pretrained(BASE_MODEL)
m0 = Matcher(BASE_MODEL, "cls").to(device)
S_dev0, S_te0, md0, mt0 = run_eval(m0, tok0)
os.makedirs(f"{OUT_ROOT}/zeroshot", exist_ok=True)
np.save(f"{OUT_ROOT}/zeroshot/test_scores.npy", S_te0)
json.dump({"model_name": BASE_MODEL, "dev": md0, "test": mt0}, open(f"{OUT_ROOT}/zeroshot/metrics.json", "w"), indent=2)
print("ZERO-SHOT dev :", {k: round(v, 4) for k, v in md0.items()})
print("ZERO-SHOT test:", {k: round(v, 4) for k, v in mt0.items()})
del m0; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 5 - FINE-TUNE the matcher on TRAM train (sampled-negative contrastive loss), seed 42
out = f"{OUT_ROOT}/ft_s42"
if os.path.exists(f"{out}/metrics.json"):
    print("skip (already done):", out)
else:
    train_matcher(BASE_MODEL, out, seed=42, epochs=8, patience=3, batch_size=16, lr=2e-5, n_neg=64, scale=30.0)

In [ ]:
# Cell 6 - results table vs the baselines (baseline numbers copied from the earlier notebooks / the paper)
def row(name, m): return {"model": name, **{k: round(m[k], 3) if m.get(k) is not None else "-" for k in ("P@1", "R@1", "MRR@3", "R@5", "tech_P@1", "seen_R@5", "unseen_R@5")}}
rows = [row("Zero-shot " + BASE_MODEL.split("/")[-1], json.load(open(f"{OUT_ROOT}/zeroshot/metrics.json"))["test"])]
p = f"{OUT_ROOT}/ft_s42/metrics.json"
if os.path.exists(p):
    r = json.load(open(p)); rows.append(row(f"Fine-tuned matcher (best epoch {r['best_epoch']})", r["test"]))
rows += [
    {"model": "TF-IDF+SVM (earlier)", "P@1": .742, "R@1": .632, "MRR@3": .814, "R@5": "-", "tech_P@1": "-", "seen_R@5": "-", "unseen_R@5": 0.0},
    {"model": "RoBERTa 3 seeds (earlier)", "P@1": .752, "R@1": .640, "MRR@3": .809, "R@5": "-", "tech_P@1": "-", "seen_R@5": "-", "unseen_R@5": 0.0},
    {"model": "Paper: Asymmetric NCE", "P@1": .770, "R@1": .658, "MRR@3": .864, "R@5": "-", "tech_P@1": .903, "seen_R@5": "-", "unseen_R@5": "-"},
]
res = pd.DataFrame(rows); print(res.to_string(index=False)); res